# OpenStack Loglarında Keşifsel Veri Analizi (EDA)

**Proje:** OpenStack Loglarında Yapay Zekâ Tabanlı Hata Tespiti ve Açıklama Doğruluğunun Değerlendirilmesi  
**Hazırlayan:** Furkan Erkarslan  
**Aşama:** Vize – veri setini tanıma ve keşifsel veri analizi

## Çalışmanın amacı

Bu notebook'ta Loghub OpenStack veri seti Python ile okunacak; ham logların yapısı, log seviyeleri, servis bileşenleri, zaman dağılımları ve sanal makine (VM) vakaları incelenecektir. Ayrıca resmî `anomaly_labels.txt` dosyasında yer alan dört enjekte hata etiketi, anormal log dosyasındaki diğer etiketsiz vakalardan ayrı tutulacaktır.

> **Kapsam notu:** Bu notebook yalnızca EDA çalışmasıdır. Isolation Forest modelinin eğitimi ve SHAP/LIME açıklamalarının karşılaştırılması bu aşamada yapılmamaktadır. Grafik ve tablolar gerçek log dosyalarından kodla üretilecektir; rastgele/varsayımsal süre verisi kullanılmayacaktır.

---

## Veri seti kaynakları

- [Loghub – OpenStack veri seti](https://github.com/logpai/loghub/tree/master/OpenStack)
- [Loghub ana deposu ve veri seti açıklaması](https://github.com/logpai/loghub)
- [OpenStack arşiv dosyası (Zenodo)](https://zenodo.org/records/8196385/files/OpenStack.tar.gz?download=1)

Veri setini kullanan çalışmalarda Loghub ve DeepLog kaynaklarına atıf verilmesi önerilmektedir. Veri setinin dağıtım koşullarını ve atıf gerekliliklerini kaynak depodan kontrol ediniz.


## 1. Gerekli kütüphaneler

Veri okuma ve düzenleme için `pandas` ve `numpy`, grafikler için `matplotlib` ve `seaborn` kullanılacaktır. Standart Python kütüphaneleri ise arşiv indirme, açma ve metin işleme işlemlerinde kullanılacaktır.

In [ ]:
import re
import tarfile
import urllib.request
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11

print("Kütüphaneler başarıyla yüklendi.")

## 2. Veri setinin indirilmesi ve dosyaların bulunması

Notebook, resmî Loghub OpenStack arşivini Zenodo bağlantısından indirmeyi dener. Arşiv daha önce indirilmişse yeniden indirilmez. İnternet bağlantısı veya indirme erişimi yoksa arşivi kendiniz indirip `data/OpenStack/` klasörüne çıkarabilir ve aşağıdaki `DATA_DIR` değişkenini dosyaların bulunduğu klasöre göre değiştirebilirsiniz.

Beklenen dosyalar:

- `openstack_normal1.log`
- `openstack_normal2.log`
- `openstack_abnormal.log`
- `anomaly_labels.txt`

**Not:** Ham log arşivi yaklaşık 59 MB'tır. İndirme ve açma işlemi internet hızına göre zaman alabilir.

In [ ]:
# Gerekirse bu yolu, log dosyalarının bulunduğu klasöre göre değiştirin.
DATA_DIR = Path("data/OpenStack")
ARCHIVE_PATH = Path("data/OpenStack.tar.gz")
EXTRACT_DIR = Path("data/extracted")

DATASET_URL = "https://zenodo.org/records/8196385/files/OpenStack.tar.gz?download=1"
REQUIRED_FILES = [
    "openstack_normal1.log",
    "openstack_normal2.log",
    "openstack_abnormal.log",
    "anomaly_labels.txt",
]


def find_dataset_folder(search_root: Path):
    """Dört beklenen veri dosyasını aynı klasörde bulursa klasörü döndürür."""
    search_root = Path(search_root)
    if not search_root.exists():
        return None
    for label_path in search_root.rglob("anomaly_labels.txt"):
        candidate = label_path.parent
        if all((candidate / name).is_file() for name in REQUIRED_FILES):
            return candidate
    # Bazı kopyalarda etiket dosyası farklı dizindeyse log dosyalarını yine bul.
    for normal_path in search_root.rglob("openstack_normal1.log"):
        candidate = normal_path.parent
        if all((candidate / name).is_file() for name in REQUIRED_FILES[:3]):
            return candidate
    return None


# Önce kullanıcının belirttiği klasör ve çalışma dizininde dosya ara.
found_dir = find_dataset_folder(DATA_DIR) or find_dataset_folder(Path.cwd())

if found_dir is None:
    ARCHIVE_PATH.parent.mkdir(parents=True, exist_ok=True)
    if not ARCHIVE_PATH.exists():
        print("OpenStack arşivi indiriliyor...")
        try:
            urllib.request.urlretrieve(DATASET_URL, ARCHIVE_PATH)
            print(f"İndirme tamamlandı: {ARCHIVE_PATH}")
        except Exception as exc:
            raise RuntimeError(
                "Otomatik indirme başarısız oldu. OpenStack.tar.gz dosyasını "
                "Loghub/Zenodo kaynağından indirip data/OpenStack/ klasörüne çıkarın. "
                "Ardından DATA_DIR değişkenini bu klasöre ayarlayıp bu hücreyi yeniden çalıştırın."
            ) from exc

    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    print("Arşiv açılıyor...")
    with tarfile.open(ARCHIVE_PATH, "r:gz") as tar:
        # Arşivden yalnızca normal dosya ve klasörleri çıkar; yol taşması ve bağlantıları atla.
        base = EXTRACT_DIR.resolve()
        for member in tar.getmembers():
            target = (EXTRACT_DIR / member.name).resolve()
            if base not in target.parents and target != base:
                continue
            if member.issym() or member.islnk():
                continue
            tar.extract(member, path=EXTRACT_DIR)

    found_dir = find_dataset_folder(EXTRACT_DIR)

if found_dir is None:
    raise FileNotFoundError(
        "Beklenen OpenStack dosyaları bulunamadı. Arşivin doğru açıldığını ve "
        "DATA_DIR yolunun doğru olduğunu kontrol edin."
    )

DATA_DIR = found_dir
print("Veri dosyalarının bulunduğu klasör:", DATA_DIR.resolve())
for name in REQUIRED_FILES:
    file_path = DATA_DIR / name
    print(f"- {name}: {'bulundu' if file_path.is_file() else 'bulunamadı'}")

## 3. Ham dosyaların ilk incelemesi

Bu bölümde her dosyanın gerçek boyutu ve satır sayısı hesaplanır. Aşağıdaki tablo, literatürde raporlanan değerleri veri setinden yeniden hesaplanan sonuçlarla karşılaştırabilmek için kullanılacaktır. Literatür değerleri analiz sonucu gibi kabul edilmeyecek; asıl sonuçlar dosyalardan hesaplanacaktır.

In [ ]:
def count_lines(file_path: Path) -> int:
    """Dosyayı belleğe tamamen almadan satır sayısını hesaplar."""
    with open(file_path, "r", encoding="utf-8", errors="replace") as f:
        return sum(1 for _ in f)


inventory_rows = []
for filename in REQUIRED_FILES[:3]:
    path = DATA_DIR / filename
    inventory_rows.append({
        "Dosya": filename,
        "Boyut (MB)": round(path.stat().st_size / (1024 ** 2), 3),
        "Ham satır": count_lines(path),
    })

inventory_df = pd.DataFrame(inventory_rows)
display(inventory_df)

print("Toplam ham log satırı:", f"{inventory_df['Ham satır'].sum():,}")
print("Etiket dosyası boyutu (KB):", round((DATA_DIR / "anomaly_labels.txt").stat().st_size / 1024, 2))

## 4. Log kayıtlarının ayrıştırılması

Loghub OpenStack satırlarında genel olarak dosya kaynağı, tarih, saat, işlem kimliği (PID), seviye, bileşen, köşeli parantez içindeki bağlam ve mesaj bulunur. Aşağıdaki düzenli ifade bu alanları çıkarmayı dener.

Her satırın başarıyla ayrıştırılması garanti değildir. Bu nedenle ayrıştırılan satır sayısı ve ayrıştırılamayan örnekler ayrıca raporlanır. Ayrıştırılamayan satırlar sessizce yok sayılmaz.

In [ ]:
LOG_PATTERN = re.compile(
    r"^\s*(?P<logrecord>\S+)\s+"
    r"(?P<date>\d{4}-\d{2}-\d{2})\s+"
    r"(?P<time>\d{2}:\d{2}:\d{2}(?:\.\d+)?)\s+"
    r"(?P<pid>\d+)\s+"
    r"(?P<level>[A-Z]+)\s+"
    r"(?P<component>\S+)\s+"
    r"\[(?P<context>.*?)\]\s*"
    r"(?P<content>.*)$"
)

UUID_PATTERN = r"[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}"
INSTANCE_CONTEXT_RE = re.compile(r"\[instance:\s*(" + UUID_PATTERN + r")\]", re.I)
SERVER_PATH_RE = re.compile(r"/servers/(" + UUID_PATTERN + r")(?:[/?\s\"']|$)", re.I)


def parse_log_file(file_path: Path) -> tuple[pd.DataFrame, list]:
    records = []
    failed_lines = []
    with open(file_path, "r", encoding="utf-8", errors="replace") as f:
        for line_number, raw_line in enumerate(f, start=1):
            raw_line = raw_line.rstrip("\n")
            match = LOG_PATTERN.match(raw_line)
            if not match:
                failed_lines.append({
                    "source_file": file_path.name,
                    "line_number": line_number,
                    "raw_line": raw_line,
                })
                continue
            row = match.groupdict()
            row["source_file"] = file_path.name
            row["line_number"] = line_number
            row["raw_line"] = raw_line
            row["timestamp"] = pd.to_datetime(
                row["date"] + " " + row["time"], errors="coerce"
            )

            # Önce açık [instance: UUID] alanını, bulunamazsa /servers/UUID yolunu ara.
            instance_match = INSTANCE_CONTEXT_RE.search(row["content"])
            if instance_match is None:
                instance_match = SERVER_PATH_RE.search(row["content"])
            row["instance_id"] = instance_match.group(1).lower() if instance_match else None
            records.append(row)

    return pd.DataFrame(records), failed_lines


parsed_frames = []
parse_failures = []
for filename in REQUIRED_FILES[:3]:
    parsed_part, failed_part = parse_log_file(DATA_DIR / filename)
    parsed_frames.append(parsed_part)
    parse_failures.extend(failed_part)

logs = pd.concat(parsed_frames, ignore_index=True)

parse_summary = pd.DataFrame([
    {"Ölçüm": "Ham satır", "Değer": int(inventory_df["Ham satır"].sum())},
    {"Ölçüm": "Ayrıştırılan satır", "Değer": len(logs)},
    {"Ölçüm": "Ayrıştırılamayan satır", "Değer": len(parse_failures)},
    {"Ölçüm": "Ayrıştırma başarısı (%)", "Değer": round(100 * len(logs) / inventory_df["Ham satır"].sum(), 2)},
    {"Ölçüm": "Zaman damgası eksik satır", "Değer": int(logs["timestamp"].isna().sum())},
])
display(parse_summary)

if parse_failures:
    print("Ayrıştırılamayan ilk 5 satır:")
    display(pd.DataFrame(parse_failures).head())
else:
    print("Tüm log satırları ayrıştırma desenine uydu.")

## 5. Log seviyeleri ve servis bileşenleri

Bu bölümde `INFO`, `WARNING`, `ERROR` gibi seviyelerin dağılımı ve en sık kayıt üreten OpenStack bileşenleri incelenir. Bu dağılımlar veri setinin genel karakterini tanımaya yardımcı olur; tek başına bir hata teşhisi veya anomali etiketi anlamına gelmez.

In [ ]:
# Log seviyelerinin sayısal dağılımı
level_counts = logs["level"].value_counts().rename_axis("Log seviyesi").reset_index(name="Kayıt sayısı")
display(level_counts)

plt.figure(figsize=(9, 5))
sns.barplot(data=level_counts, x="Log seviyesi", y="Kayıt sayısı")
plt.title("OpenStack Log Seviyelerinin Dağılımı")
plt.xlabel("Log seviyesi")
plt.ylabel("Kayıt sayısı")
plt.tight_layout()
plt.show()

# En sık görülen bileşenler
top_components = (
    logs["component"].value_counts().head(15)
    .rename_axis("Bileşen").reset_index(name="Kayıt sayısı")
)
display(top_components)

plt.figure(figsize=(10, 6))
sns.barplot(data=top_components, y="Bileşen", x="Kayıt sayısı")
plt.title("En Çok Log Üreten İlk 15 OpenStack Bileşeni")
plt.xlabel("Kayıt sayısı")
plt.ylabel("Bileşen")
plt.tight_layout()
plt.show()

## 6. Zaman dağılımı

Zaman damgaları üzerinden kayıtların saatlere göre dağılımı incelenir. Bu grafik, logların hangi zaman aralığında yoğunlaştığını gösterir. Zaman yoğunluğu tek başına arıza göstergesi olarak yorumlanmamalıdır.

In [ ]:
time_logs = logs.dropna(subset=["timestamp"]).copy()
time_logs["Saat"] = time_logs["timestamp"].dt.hour
hour_counts = time_logs["Saat"].value_counts().sort_index().reindex(range(24), fill_value=0)

plt.figure(figsize=(11, 5))
sns.barplot(x=hour_counts.index, y=hour_counts.values)
plt.title("Saatlere Göre OpenStack Log Kayıt Sayısı")
plt.xlabel("Saat")
plt.ylabel("Kayıt sayısı")
plt.xticks(range(24))
plt.tight_layout()
plt.show()

print("İlk zaman damgası:", time_logs["timestamp"].min())
print("Son zaman damgası:", time_logs["timestamp"].max())
print("Zaman damgası bulunan kayıt:", f"{len(time_logs):,}")

## 7. Resmî anomali etiketlerinin okunması

`anomaly_labels.txt` dosyası, hata enjeksiyonu yapılan dört VM örneğinin kimliklerini içerir. Etiketler bu dosyadan okunur. Anormal log dosyasında bulunan ancak bu listede yer almayan diğer vakalar **normal kabul edilmez**; bunlar “etiketsiz anormal dosya vakası” olarak ayrı bir grupta tutulur.

Bu ayrım, sınıf dağılımını yanlış yorumlamamak için önemlidir.

In [ ]:
label_text = (DATA_DIR / "anomaly_labels.txt").read_text(encoding="utf-8", errors="replace")
anomaly_ids = set(x.lower() for x in re.findall(UUID_PATTERN, label_text))

print("Etiket dosyasında bulunan benzersiz VM kimliği:", len(anomaly_ids))
print("Etiket dosyası örneği:")
print("\n".join(label_text.splitlines()[:10]))

if len(anomaly_ids) != 4:
    print("UYARI: Etiket dosyasında dört benzersiz kimlik dışında bir sayı bulundu. Dosyanın içeriğini kontrol edin.")

## 8. VM vakalarının oluşturulması

Aynı log dosyasında aynı VM kimliğiyle ilişkilendirilen kayıtlar bir vaka altında toplanır. Dosya adı vaka anahtarına dâhil edilir; böylece farklı log dosyalarında aynı kimlik bulunsa bile vakalar birbirine karıştırılmaz.

Her vaka için şu temel bilgiler hesaplanır:

- Log kayıt sayısı
- İlk ve son kayıt zamanı
- Vaka süresi
- `Creating image` ve `VM Started` olaylarının bulunup bulunmadığı
- Bu iki olay arasındaki süre (hesaplanabiliyorsa)
- Vakanın etiket durumu

`/servers/detail` sorguları belirli bir VM'ye doğrudan ait olmayabildiğinden, ayrıca vakanın zaman aralığında aynı dosyada kaç kez geçtiği bağlamsal bir sayaç olarak hesaplanır.

In [ ]:
linked_logs = logs.dropna(subset=["instance_id", "timestamp"]).copy()
linked_logs["case_id"] = linked_logs["source_file"] + "::" + linked_logs["instance_id"]

# Her dosyada /servers/detail içeren kayıtların zamanlarını hazırla.
detail_times_by_file = {}
for filename, part in logs.dropna(subset=["timestamp"]).groupby("source_file"):
    mask = part["content"].str.contains(r"/servers/detail", case=False, regex=True, na=False)
    detail_times_by_file[filename] = np.sort(part.loc[mask, "timestamp"].astype("int64").to_numpy())

case_rows = []
for (source_file, instance_id), group in linked_logs.groupby(["source_file", "instance_id"]):
    group = group.sort_values("timestamp")
    timestamps = group["timestamp"].dropna()
    first_time = timestamps.min()
    last_time = timestamps.max()

    content = group["content"].fillna("")
    creating_times = group.loc[
        content.str.contains(r"Creating image", case=False, regex=True), "timestamp"
    ].dropna().sort_values()
    started_times = group.loc[
        content.str.contains(r"VM Started", case=False, regex=True), "timestamp"
    ].dropna().sort_values()

    image_to_start = np.nan
    if len(creating_times) and len(started_times):
        first_image = creating_times.iloc[0]
        starts_after_image = started_times[started_times >= first_image]
        if len(starts_after_image):
            image_to_start = (starts_after_image.iloc[0] - first_image).total_seconds()

    if source_file in ("openstack_normal1.log", "openstack_normal2.log"):
        label_status = "Normal dosyadan gelen vaka"
    elif instance_id in anomaly_ids:
        label_status = "Etiketli enjekte anomali"
    else:
        label_status = "Anormal dosyada etiketsiz vaka"

    detail_times = detail_times_by_file.get(source_file, np.array([], dtype="int64"))
    start_ns = first_time.value
    end_ns = last_time.value
    left = np.searchsorted(detail_times, start_ns, side="left")
    right = np.searchsorted(detail_times, end_ns, side="right")

    case_rows.append({
        "source_file": source_file,
        "instance_id": instance_id,
        "case_id": source_file + "::" + instance_id,
        "label_status": label_status,
        "event_count": len(group),
        "first_timestamp": first_time,
        "last_timestamp": last_time,
        "case_duration_sec": (last_time - first_time).total_seconds(),
        "creating_image_count": int(content.str.contains(r"Creating image", case=False, regex=True).sum()),
        "vm_started_count": int(content.str.contains(r"VM Started", case=False, regex=True).sum()),
        "creating_image_to_vm_started_sec": image_to_start,
        "servers_detail_in_case_window": int(right - left),
    })

cases = pd.DataFrame(case_rows)
print("VM kimliğiyle ilişkilendirilen log satırı:", f"{len(linked_logs):,}")
print("Oluşturulan dosya-kapsamlı VM vakası:", f"{len(cases):,}")
display(cases.head())

## 9. Vaka ve etiket dağılımı

Bu grafikte üç kategori gösterilir: normal dosyalardan gelen vakalar, anormal dosyada olup resmî etiketi bulunmayan vakalar ve resmî olarak etiketlenmiş dört enjekte anomali. Etiketsiz 194 vaka “normal” olarak yeniden etiketlenmemektedir.

In [ ]:
label_order = [
    "Normal dosyadan gelen vaka",
    "Anormal dosyada etiketsiz vaka",
    "Etiketli enjekte anomali",
]
case_distribution = (
    cases["label_status"].value_counts()
    .reindex(label_order, fill_value=0)
    .rename_axis("Vaka grubu").reset_index(name="Vaka sayısı")
)
display(case_distribution)

plt.figure(figsize=(10, 5))
ax = sns.barplot(data=case_distribution, x="Vaka grubu", y="Vaka sayısı")
plt.title("OpenStack VM Vakalarının Etiket Durumuna Göre Dağılımı")
plt.xlabel("")
plt.ylabel("Vaka sayısı")
plt.xticks(rotation=12, ha="right")
for container in ax.containers:
    ax.bar_label(container, fmt="%d", padding=3)
plt.tight_layout()
plt.show()

print("Not: Anormal dosyadaki etiketsiz vakalar normal olarak değerlendirilmemiştir.")

## 10. VM oluşturma aşamasındaki gerçek sürelerin incelenmesi

Önceki taslaktaki rastgele süre üreten örnek kaldırılmıştır. Bu bölümde `Creating image` ve onu takip eden ilk `VM Started` loglarının gerçek zaman damgaları arasındaki fark hesaplanır.

Yalnızca iki olay da bulunup süre hesaplanabilen vakalar grafiğe dâhil edilir. Dört etiketli vakanın sayısı az olduğundan sonuçlar dikkatli yorumlanmalıdır.

In [ ]:
delay_cases = cases.dropna(subset=["creating_image_to_vm_started_sec"]).copy()

if delay_cases.empty:
    print("Bu log ayrıştırmasıyla Creating image → VM Started süreleri hesaplanamadı.")
else:
    delay_summary = (
        delay_cases.groupby("label_status")["creating_image_to_vm_started_sec"]
        .agg(["count", "mean", "median", "min", "max"])
        .round(3)
    )
    display(delay_summary)

    plt.figure(figsize=(10, 5))
    sns.boxplot(
        data=delay_cases,
        x="label_status",
        y="creating_image_to_vm_started_sec",
        showfliers=True,
    )
    sns.stripplot(
        data=delay_cases,
        x="label_status",
        y="creating_image_to_vm_started_sec",
        color="black",
        alpha=0.45,
        size=3,
    )
    plt.title("Gerçek Loglardan Hesaplanan Creating image → VM Started Süresi")
    plt.xlabel("")
    plt.ylabel("Süre (saniye)")
    plt.xticks(rotation=12, ha="right")
    plt.tight_layout()
    plt.show()

    print("Grafik yalnızca iki yaşam döngüsü olayı da bulunan vakaları içerir.")

## 11. Vaka süresi ve `/servers/detail` sorguları

Bu bölüm, her vaka zaman aralığında toplam log süresini ve aynı dosyada gözlenen `/servers/detail` sorgu yoğunluğunu karşılaştırır. Sorgular bir VM'ye doğrudan ait olmayabileceği için bu sayaç **bağlamsal bir zaman-penceresi ölçüsüdür**; tek başına VM'nin sorguyu yaptığı anlamına gelmez.

In [ ]:
for feature, title, ylabel in [
    ("case_duration_sec", "VM Vakalarının Loglarda Gözlenen Süresi", "Süre (saniye)"),
    ("servers_detail_in_case_window", "Vaka Zaman Aralığındaki /servers/detail Sorguları", "Sorgu sayısı"),
]:
    plot_df = cases.dropna(subset=[feature]).copy()
    if plot_df.empty:
        print(f"{feature} için çizilecek veri bulunamadı.")
        continue
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=plot_df, x="label_status", y=feature, showfliers=True)
    sns.stripplot(data=plot_df, x="label_status", y=feature, color="black", alpha=0.35, size=2.5)
    plt.title(title)
    plt.xlabel("")
    plt.ylabel(ylabel)
    plt.xticks(rotation=12, ha="right")
    plt.tight_layout()
    plt.show()

    display(plot_df.groupby("label_status")[feature].agg(["count", "mean", "median", "min", "max"]).round(3))

## 12. Etiketli anomali vakalarının ayrı incelenmesi

Aşağıdaki tablo yalnızca resmî etiket dosyasında bulunan dört VM kimliğini gösterir. Kimliklerin her birinin anormal log dosyasında bulunup bulunmadığı ve temel süre ölçümleri kontrol edilir.

In [ ]:
labeled_cases = cases[cases["label_status"] == "Etiketli enjekte anomali"].copy()

if labeled_cases.empty:
    print("Etiket dosyasındaki VM kimlikleri vaka tablosuyla eşleşmedi. Kimlik çıkarma adımını kontrol edin.")
else:
    display(labeled_cases[[
        "instance_id",
        "event_count",
        "case_duration_sec",
        "creating_image_count",
        "vm_started_count",
        "creating_image_to_vm_started_sec",
        "servers_detail_in_case_window",
    ]].sort_values("creating_image_to_vm_started_sec", ascending=False))

print("Etiket dosyasındaki kimlik sayısı:", len(anomaly_ids))
print("Anormal log dosyasında eşleşen etiketli vaka sayısı:", len(labeled_cases))

## 13. EDA bulgularının otomatik özeti

Bu hücre, çalıştırılan gerçek veriden temel sayıları özetler. Sonuçları rapora aktarırken burada üretilen sayıları kullanın; literatürdeki değerlerle farklılık varsa ayrıştırma ve vaka oluşturma kurallarını kontrol edin.

In [ ]:
total_raw = int(inventory_df["Ham satır"].sum())
parsed_count = len(logs)
linked_count = len(linked_logs)

print("EDA ÖZETİ")
print("-" * 55)
print(f"Ham log satırı                 : {total_raw:,}")
print(f"Ayrıştırılan log satırı        : {parsed_count:,}")
print(f"Ayrıştırılamayan log satırı    : {len(parse_failures):,}")
print(f"VM kimliğiyle eşleşen satır   : {linked_count:,}")
print(f"Oluşturulan VM vakası         : {len(cases):,}")
print(f"Resmî etiketli anomali kimliği: {len(anomaly_ids):,}")
print()
print("Vaka grupları:")
for _, row in case_distribution.iterrows():
    print(f"- {row['Vaka grubu']}: {int(row['Vaka sayısı']):,}")

if len(anomaly_ids) != 4 or len(labeled_cases) != 4:
    print("\nKONTROL: Beklenen dört resmî anomali kimliğinin tümü vaka tablosuyla eşleşmedi.")
else:
    print("\nKontrol: Dört resmî anomali kimliği de anormal dosyada vaka olarak bulundu.")

## 14. Değerlendirme ve sınırlılıklar

Bu notebook'un sonuçları, dosyaların başarıyla indirilip ayrıştırılması ve yukarıdaki hücrelerin sırasıyla çalıştırılması sonrasında oluşacaktır. Sonuçları rapora aktarırken aşağıdaki noktaları dikkate alın:

1. Resmî etiket dosyasında yalnızca dört enjekte anomali kimliği bulunmaktadır. Anormal dosyada yer alan diğer vakalar etiketsizdir; bu nedenle normal sınıfına eklenmemelidir.
2. `Creating image` → `VM Started` süresi yalnızca iki olayın da bulunduğu vakalarda hesaplanır. Olaylardan biri yoksa değer boş bırakılır; sıfır ile doldurulmaz.
3. `/servers/detail` sayısı vaka zaman aralığındaki dosya düzeyindeki sorgu sayısıdır. Bu sorgu sayısı doğrudan ilgili VM'ye ait bir sayaç olarak yorumlanmamalıdır.
4. Dört etiketli örnek, genelleme yapmak için küçük bir sayıdır. Grafikler keşif ve veri tanıma amacı taşır; nihai model başarısı göstermez.
5. Bu aşamada Isolation Forest, SHAP veya LIME eğitimi/uygulaması yapılmamıştır. Bunlar projenin sonraki aşamasıdır.

### Sonraki aşama

EDA bulguları doğrulandıktan sonra VM-vaka düzeyinde özellik matrisi hazırlanacak; ardından Isolation Forest ile anomali puanları üretilecek ve SHAP/LIME açıklamaları değerlendirilecektir.

## Kaynaklar

1. LogPAI, [Loghub – OpenStack Dataset](https://github.com/logpai/loghub/tree/master/OpenStack).
2. M. Du, F. Li, G. Zheng, V. Srikumar, “DeepLog: Anomaly Detection and Diagnosis from System Logs through Deep Learning,” ACM CCS, 2017. [Makale](https://doi.org/10.1145/3133956.3134015).
3. J. Zhu et al., “Loghub: A Large Collection of System Log Datasets for AI-driven Log Analytics,” IEEE ISSRE, 2023. [Loghub deposu](https://github.com/logpai/loghub).
4. Q. Xin, “Explaining OpenStack Failure-Injection Log Anomalies with Retrieved Normal Prototypes,” 2025. [Makale sayfası](https://doi.org/10.18196/eist.v6i2.31232).

**Veri kullanım notu:** Ham veri setini yeniden dağıtmadan önce Loghub deposundaki güncel kullanım ve atıf koşullarını inceleyin. Notebook, veri dosyalarının kaynağını belirtir ve veriyi çalıştırma sırasında resmî arşivden edinir.